# Physics-Augmented AI-Image Detection — Notebook 03: compression-aware training

Notebook 02 found that the physics features barely change under JPEG compression (AUC 0.68 → 0.67 at q50),
while CLIP drops from 0.97 to 0.79. But a fusion trained on clean images keeps trusting CLIP.

This notebook gives every model a fair chance to handle compression:
1. Features for the **training** images are recomputed after a random JPEG compression (quality 50–95, fixed per image).
2. **CLIP**, **physics** and the **fusion** are trained on clean + compressed copies. The **quality-aware fusion**
   also gets the JPEG quality, so its trust in each branch can change with compression.
3. Test: the same clean / JPEG q75 / JPEG q50 / 0.5× downscale test sets as notebook 02 (features reused from its output).
   Resizing is **never** used in training, so `down0.5` is a held-out kind of processing.
4. **Paired bootstrap 95% confidence intervals** for every difference that matters.

Leakage guard: the quality value given to the fusion is the quality of the processing *we* applied
(96 = no extra processing). It never comes from the original files, so it cannot encode a real/fake difference.

**Kaggle setup:** *Add Input* → notebook **01** output, notebook **02** output, and the **Unbiased Tiny GenImage** dataset.
*GPU T4*, *Internet On*, then *Save Version → Save & Run All*. Expected ≈ 1.5–2 h (normals for ~6.7k training images).

In [ ]:
import os, sys, json, time, glob, warnings
from pathlib import Path

IN_KAGGLE = os.path.exists("/kaggle/input")
OUT_DIR = Path(os.environ.get("PHYSFAKE_OUT", "/kaggle/working" if IN_KAGGLE else "./work03")).resolve()
INPUT_ROOT = os.environ.get("PHYSFAKE_INPUT_ROOT", "/kaggle/input")
NB01_DIR = os.environ.get("PHYSFAKE_NB01_DIR", "")
NB02_DIR = os.environ.get("PHYSFAKE_NB02_DIR", "")
CLIP_MODEL = os.environ.get("PHYSFAKE_CLIP_MODEL", "openai/clip-vit-large-patch14")
NORMAL_MODEL = os.environ.get("PHYSFAKE_NORMAL_MODEL", "") or None
PERTURBATIONS = ["jpeg75", "jpeg50", "down0.5"]
NORMALISE_FORMAT = True
NORMAL_BATCH = 8
N_BOOT = int(os.environ.get("PHYSFAKE_N_BOOT", "500"))
N_CPU = os.cpu_count() or 2
CHUNK = 1000
SEED = 0
FEAT_DIR = OUT_DIR / "features"
for d in (OUT_DIR, FEAT_DIR, OUT_DIR / "physfeat", OUT_DIR / "models"):
    d.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(OUT_DIR))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
print("output:", OUT_DIR)

## 1 · Feature-extraction package

In [ ]:
%%writefile {OUT_DIR}/physfeat/__init__.py
"""physfeat — physics-based features for AI-generated image detection."""

In [ ]:
%%writefile {OUT_DIR}/physfeat/imaging.py
"""Image I/O shared by every feature extractor.

* `read_meta`      – format, size, JPEG quality, hash (for the bias audit / dedupe)
* `load_rgb`       – decode an image the same way for every extractor, optionally
                     neutralising the PNG-vs-JPEG shortcut and applying a
                     robustness perturbation (JPEG re-compression, resizing, blur).
"""
from __future__ import annotations

import hashlib
import io

import numpy as np
from PIL import Image, ImageFile, ImageFilter

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = None

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff", ".jfif"}
NORMALISE_JPEG_Q = 96  # GenImage de-biasing protocol (Grommelt et al., 2024)

# ----------------------------------------------------------------------------
# JPEG quality estimation (compare the luminance table with Pillow's own tables)
# ----------------------------------------------------------------------------
_QREF = None


def _qref():
    global _QREF
    if _QREF is None:
        tiny = Image.new("RGB", (16, 16), (128, 128, 128))
        ref = {}
        for q in range(1, 101):
            buf = io.BytesIO()
            tiny.save(buf, "JPEG", quality=q)
            ref[q] = np.array(Image.open(io.BytesIO(buf.getvalue())).quantization[0], float)
        _QREF = ref
    return _QREF


def estimate_jpeg_quality(im: Image.Image) -> float:
    q = getattr(im, "quantization", None)
    if not q or 0 not in q:
        return float("nan")
    t = np.array(q[0], float)
    ref = _qref()
    return float(min(ref, key=lambda k: np.abs(ref[k] - t).mean()))


def file_md5(path, chunk=1 << 20) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def read_meta(path) -> dict:
    out = {"fmt": None, "width": np.nan, "height": np.nan, "jpeg_q": np.nan,
           "bytes": np.nan, "md5": None, "meta_error": None}
    try:
        import os
        out["bytes"] = os.path.getsize(path)
        out["md5"] = file_md5(path)
        with Image.open(path) as im:
            out["fmt"] = im.format
            out["width"], out["height"] = im.size
            if im.format == "JPEG":
                out["jpeg_q"] = estimate_jpeg_quality(im)
    except Exception as e:  # corrupt / unreadable
        out["meta_error"] = repr(e)[:200]
    return out


# ----------------------------------------------------------------------------
# decoding + perturbations
# ----------------------------------------------------------------------------
def jpeg_roundtrip(im: Image.Image, quality: int) -> Image.Image:
    buf = io.BytesIO()
    im.save(buf, "JPEG", quality=quality)
    return Image.open(io.BytesIO(buf.getvalue())).convert("RGB")


def _resize(im, s):
    w, h = im.size
    return im.resize((max(8, round(w * s)), max(8, round(h * s))), Image.BICUBIC)


PERTURBATIONS = {
    None: lambda im: im,
    "jpeg90": lambda im: jpeg_roundtrip(im, 90),
    "jpeg75": lambda im: jpeg_roundtrip(im, 75),
    "jpeg50": lambda im: jpeg_roundtrip(im, 50),
    "down0.5": lambda im: _resize(im, 0.5),
    "blur1": lambda im: im.filter(ImageFilter.GaussianBlur(1.0)),
}


JPEGRAND_RANGE = (50, 95)   # training-time augmentation: per-image JPEG quality, deterministic per file


def jpegrand_quality(path) -> int:
    import os
    lo, hi = JPEGRAND_RANGE
    return lo + stable_seed(os.path.basename(str(path))) % (hi - lo + 1)


def effective_quality(perturb, original_q=float("nan")) -> float:
    """JPEG quality of the image the detector actually sees (what a deployed system
    reads from the file header). Resizing/blur keep the original quality."""
    if perturb and perturb.startswith("jpeg") and perturb[4:].isdigit():
        return float(perturb[4:])
    q = original_q
    return float(NORMALISE_JPEG_Q if not np.isfinite(q) else q)


def load_pil(path, normalise_format: bool = True, perturb: str | None = None) -> Image.Image:
    with Image.open(path) as im:
        fmt = im.format
        im = im.convert("RGB")
    if normalise_format and fmt != "JPEG":
        # PNG fakes vs JPEG reals is a known dataset shortcut: give every image a
        # JPEG history so the detector cannot key on the container format.
        im = jpeg_roundtrip(im, NORMALISE_JPEG_Q)
    if perturb == "jpegrand":
        return jpeg_roundtrip(im, jpegrand_quality(path))
    return PERTURBATIONS[perturb](im)


def load_rgb(path, normalise_format: bool = True, perturb: str | None = None) -> np.ndarray:
    return np.asarray(load_pil(path, normalise_format, perturb))


def stable_seed(path) -> int:
    return int(hashlib.md5(str(path).encode()).hexdigest()[:8], 16)

In [ ]:
%%writefile {OUT_DIR}/physfeat/geometry.py
"""
Projective-geometry (perspective) physics features for AI-generated image detection.

Idea
----
In a real photograph taken with a (roughly) pinhole camera, images of parallel 3-D
lines converge *exactly* to a common vanishing point (VP), and the VPs of mutually
orthogonal directions obey camera constraints (principal point near the image
centre, positive focal length). Generators learn perspective only statistically,
so their lines tend to *almost* converge: scattered intersections, one VP split
into several nearby ones, and VP triplets that imply an impossible camera.

Pipeline
--------
1. Resize (max side = 512 by default) -> grayscale -> LSD line segments.
2. Sequential, length-weighted RANSAC to find up to 3 dominant VPs
   (VPs may be finite or at infinity; everything is done in homogeneous coords).
3. Least-squares refinement of each VP, then measurement of how well its
   inlier segments actually converge (angular residuals).
4. Camera-consistency checks between VPs (implied focal length, principal point).

All coordinates are normalised: origin at the image centre, unit = half diagonal.
Features are NaN when they are undefined (e.g. no lines in a photo of a cat);
downstream models must handle NaN (HistGradientBoosting does natively).
"""
from __future__ import annotations

import hashlib
from dataclasses import dataclass, field

import cv2
import numpy as np

# ----------------------------------------------------------------------------
# configuration
# ----------------------------------------------------------------------------
MAX_SIDE = 512            # images are resized so the longer side is this many px
MIN_LEN_FRAC = 0.025      # keep segments longer than this fraction of the diagonal
LONG_LEN_FRAC = 0.08      # "long" segments (used for the unexplained-lines feature)
INLIER_DEG = 1.5          # a segment supports a VP if its angular error is below this
TIGHT_DEG = 0.5           # threshold for the "tight convergence" fraction
N_HYP = 400               # RANSAC hypotheses per VP
MAX_VPS = 3
MIN_VP_SUPPORT = 4        # minimum number of inlier segments for a VP
MIN_VP_LEN_FRAC = 0.04    # minimum share of total line length for a VP
MAX_SEGMENTS = 1500       # cap (longest kept) to bound runtime on very busy images
FINITE_LIMIT = 8.0        # |VP| beyond this many half-diagonals is treated as "at infinity"

FEATURE_NAMES = [
    "geo_log_nseg",          # log(1 + number of segments kept)
    "geo_line_density",      # total segment length / image diagonal
    "geo_n_vps",             # number of VPs found (0..3)
    "geo_frac_vp1", "geo_frac_vp2", "geo_frac_vp3",   # share of line length supporting each VP
    "geo_frac_long_unexpl",  # share of long-segment length explained by no VP
    "geo_res_vp1", "geo_res_vp2", "geo_res_vp3",      # length-weighted mean angular residual (deg)
    "geo_tight_vp1", "geo_tight_vp2", "geo_tight_vp3",  # share of inlier length with residual < TIGHT_DEG
    "geo_cv_res_vp1",        # split-half cross-validated residual of VP1 (deg)
    "geo_min_vp_sep_deg",    # smallest angle between VP directions (split VPs are suspicious)
    "geo_ortho_cos_min",     # best |cos| between VP directions under the assumed camera
    "geo_f2_12",             # signed log focal^2 implied by VP1-VP2 orthogonality
    "geo_pp_offset",         # principal point (orthocentre of 3 finite VPs) distance from centre
]


@dataclass
class GeometryResult:
    features: dict
    segments: np.ndarray = field(default=None, repr=False)   # (N,4) pixel coords in resized image
    labels: np.ndarray = field(default=None, repr=False)     # (N,) VP index or -1
    residuals: np.ndarray = field(default=None, repr=False)  # (N,) angular error to own VP (deg)
    vps_px: list = field(default_factory=list)                # VPs in resized-pixel homogeneous coords
    image_rgb: np.ndarray = field(default=None, repr=False)  # resized RGB image (for plotting)


# ----------------------------------------------------------------------------
# image loading
# ----------------------------------------------------------------------------
def load_image(path_or_array, max_side: int = MAX_SIDE) -> np.ndarray:
    """Return an RGB uint8 image whose longer side is `max_side`."""
    if isinstance(path_or_array, np.ndarray):
        rgb = path_or_array
    else:
        bgr = cv2.imread(str(path_or_array), cv2.IMREAD_COLOR)
        if bgr is None:  # some formats (e.g. certain webp/gif) -> fall back to PIL
            from PIL import Image
            rgb = np.array(Image.open(path_or_array).convert("RGB"))
        else:
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    if rgb.ndim == 2:
        rgb = np.stack([rgb] * 3, -1)
    h, w = rgb.shape[:2]
    s = max_side / max(h, w)
    if abs(s - 1) > 1e-3:
        interp = cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC
        rgb = cv2.resize(rgb, (round(w * s), round(h * s)), interpolation=interp)
    return rgb


# ----------------------------------------------------------------------------
# core geometry helpers
# ----------------------------------------------------------------------------
_LSD = None


def _lsd():
    global _LSD
    if _LSD is None:  # one detector per process
        _LSD = cv2.createLineSegmentDetector(cv2.LSD_REFINE_STD)
    return _LSD


def detect_segments(gray: np.ndarray) -> np.ndarray:
    h, w = gray.shape
    diag = np.hypot(h, w)
    lines = _lsd().detect(gray)[0]
    if lines is None:
        return np.zeros((0, 4), np.float64)
    segs = lines.reshape(-1, 4).astype(np.float64)
    L = np.hypot(segs[:, 2] - segs[:, 0], segs[:, 3] - segs[:, 1])
    keep = L >= MIN_LEN_FRAC * diag
    segs, L = segs[keep], L[keep]
    if len(segs) > MAX_SEGMENTS:
        segs = segs[np.argsort(-L)[:MAX_SEGMENTS]]
    return segs


def _normalise(segs, h, w):
    c = np.array([w / 2.0, h / 2.0])
    s = np.hypot(h, w) / 2.0
    p1 = (segs[:, :2] - c) / s
    p2 = (segs[:, 2:] - c) / s
    return p1, p2, c, s


def _angles_to_vps(mid, d, vps):
    """Angular error (deg, 0..90) between each segment and the line joining its
    midpoint to each VP.  vps: (K,3) homogeneous.  Returns (K,N)."""
    # direction from midpoint towards VP; valid for finite and infinite VPs
    dv = vps[:, None, :2] - mid[None] * vps[:, None, 2:3]          # (K,N,2)
    num = np.abs(dv[..., 0] * d[None, :, 0] + dv[..., 1] * d[None, :, 1])
    den = np.linalg.norm(dv, axis=-1) * np.linalg.norm(d, axis=-1)[None] + 1e-12
    return np.degrees(np.arccos(np.clip(num / den, 0.0, 1.0)))


def _refine_vp(lines_n, weights):
    """Weighted least squares VP: argmin_{|v|=1} sum w_i (l_i . v)^2."""
    A = lines_n * np.sqrt(weights)[:, None]
    _, _, vt = np.linalg.svd(A, full_matrices=False)
    v = vt[-1]
    return v / (np.linalg.norm(v) + 1e-12)


def _wmean(x, w):
    return float(np.sum(x * w) / (np.sum(w) + 1e-12))


def _is_finite(v):
    return abs(v[2]) > 1e-9 and np.linalg.norm(v[:2] / v[2]) < FINITE_LIMIT


def _ray(v):
    """3-D direction of a VP under a camera with principal point at the centre and
    focal length = 1 half-diagonal (~53 deg diagonal FOV); used only for
    separation angles, not for the camera-consistency tests."""
    r = np.array([v[0], v[1], v[2]], float)
    return r / (np.linalg.norm(r) + 1e-12)


# ----------------------------------------------------------------------------
# main entry point
# ----------------------------------------------------------------------------
def extract(path_or_array, seed: int | None = None, keep_debug: bool = False) -> GeometryResult:
    rgb = load_image(path_or_array)
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape
    diag = np.hypot(h, w)
    feats = {k: np.nan for k in FEATURE_NAMES}

    segs = detect_segments(gray)
    n = len(segs)
    feats["geo_log_nseg"] = float(np.log1p(n))
    L = np.hypot(segs[:, 2] - segs[:, 0], segs[:, 3] - segs[:, 1]) if n else np.zeros(0)
    feats["geo_line_density"] = float(L.sum() / diag)
    feats["geo_n_vps"] = 0.0
    labels = -np.ones(n, int)
    resid = np.full(n, np.nan)
    vps = []

    if n >= 2 * MIN_VP_SUPPORT:
        if seed is None:
            seed = int(hashlib.md5(str(path_or_array if not isinstance(path_or_array, np.ndarray)
                                        else path_or_array.shape).encode()).hexdigest()[:8], 16)
        rng = np.random.default_rng(seed)
        p1, p2, _, _ = _normalise(segs, h, w)
        mid, d = (p1 + p2) / 2, p2 - p1
        Ln = np.linalg.norm(d, axis=1)
        lines = np.cross(np.c_[p1, np.ones(n)], np.c_[p2, np.ones(n)])
        lines /= np.linalg.norm(lines[:, :2], axis=1, keepdims=True) + 1e-12  # (a,b) unit
        total_len = Ln.sum()
        free = np.ones(n, bool)

        for k in range(MAX_VPS):
            idx = np.flatnonzero(free)
            if len(idx) < MIN_VP_SUPPORT:
                break
            prob = Ln[idx] / Ln[idx].sum()
            i = rng.choice(idx, N_HYP, p=prob)
            j = rng.choice(idx, N_HYP, p=prob)
            ok = i != j
            hyp = np.cross(lines[i[ok]], lines[j[ok]])
            nrm = np.linalg.norm(hyp, axis=1)
            hyp = hyp[nrm > 1e-9] / nrm[nrm > 1e-9, None]
            if len(hyp) == 0:
                break
            ang = _angles_to_vps(mid[idx], d[idx], hyp)          # (H, |idx|)
            score = ((ang < INLIER_DEG) * Ln[idx][None]).sum(1)
            v = hyp[np.argmax(score)]
            # refine twice (inliers -> LS fit -> inliers)
            for _ in range(2):
                a = _angles_to_vps(mid[idx], d[idx], v[None])[0]
                inl = idx[a < INLIER_DEG]
                if len(inl) < 2:
                    break
                v = _refine_vp(lines[inl], Ln[inl])
            a_all = _angles_to_vps(mid, d, v[None])[0]
            inl = idx[a_all[idx] < INLIER_DEG]
            if len(inl) < MIN_VP_SUPPORT or Ln[inl].sum() < MIN_VP_LEN_FRAC * total_len:
                break
            labels[inl] = k
            resid[inl] = a_all[inl]
            free[inl] = False
            vps.append(v)

            kk = k + 1
            feats[f"geo_frac_vp{kk}"] = float(Ln[inl].sum() / total_len)
            feats[f"geo_res_vp{kk}"] = _wmean(a_all[inl], Ln[inl])
            feats[f"geo_tight_vp{kk}"] = float(Ln[inl][a_all[inl] < TIGHT_DEG].sum() / Ln[inl].sum())
            if k == 0 and len(inl) >= 2 * MIN_VP_SUPPORT:
                # split-half cross-validation: fit on one half, measure the other
                perm = rng.permutation(inl)
                A, B = perm[: len(perm) // 2], perm[len(perm) // 2:]
                vA, vB = _refine_vp(lines[A], Ln[A]), _refine_vp(lines[B], Ln[B])
                eAB = _wmean(_angles_to_vps(mid[B], d[B], vA[None])[0], Ln[B])
                eBA = _wmean(_angles_to_vps(mid[A], d[A], vB[None])[0], Ln[A])
                feats["geo_cv_res_vp1"] = 0.5 * (eAB + eBA)

        feats["geo_n_vps"] = float(len(vps))
        long_ = Ln >= (LONG_LEN_FRAC * diag) / (diag / 2)
        if long_.any():
            feats["geo_frac_long_unexpl"] = float(Ln[long_ & (labels < 0)].sum() / Ln[long_].sum())

        # ---- relations between VPs --------------------------------------------------
        if len(vps) >= 2:
            rays = [_ray(v) for v in vps]
            seps, coss = [], []
            for a in range(len(vps)):
                for b in range(a + 1, len(vps)):
                    c = abs(float(np.dot(rays[a], rays[b])))
                    seps.append(np.degrees(np.arccos(min(1.0, c))))
                    coss.append(c)
            feats["geo_min_vp_sep_deg"] = float(min(seps))
            feats["geo_ortho_cos_min"] = float(min(coss))

            v1, v2 = vps[0], vps[1]
            if _is_finite(v1) and _is_finite(v2):
                # orthogonality with principal point at centre:
                # f^2 = -(x1 x2 + y1 y2) for VPs in normalised inhomogeneous coords
                q1, q2 = v1[:2] / v1[2], v2[:2] / v2[2]
                f2 = -float(np.dot(q1, q2))
                feats["geo_f2_12"] = float(np.sign(f2) * np.log1p(abs(f2)))

        if len(vps) == 3 and all(_is_finite(v) for v in vps):
            # principal point = orthocentre of the VP triangle (3 orthogonal directions)
            q = [v[:2] / v[2] for v in vps]
            A = np.array([q[2] - q[1], q[2] - q[0]])
            b = np.array([np.dot(q[0], q[2] - q[1]), np.dot(q[1], q[2] - q[0])])
            if abs(np.linalg.det(A)) > 1e-9:
                pp = np.linalg.solve(A, b)
                feats["geo_pp_offset"] = float(min(np.linalg.norm(pp), 10.0))

    res = GeometryResult(features=feats)
    if keep_debug:
        c = np.array([w / 2.0, h / 2.0]); s = diag / 2.0
        T = np.array([[s, 0, c[0]], [0, s, c[1]], [0, 0, 1.0]])   # normalised -> pixel
        res.segments, res.labels, res.residuals = segs, labels, resid
        res.vps_px = [T @ v for v in vps]
        res.image_rgb = rgb
    return res


def extract_features(path) -> dict:
    """Picklable wrapper for multiprocessing pools; never raises."""
    try:
        return extract(path).features
    except Exception as e:  # corrupt file etc.
        out = {k: np.nan for k in FEATURE_NAMES}
        out["geo_error"] = repr(e)[:200]
        return out


# ----------------------------------------------------------------------------
# visualisation (used by notebooks and the demo)
# ----------------------------------------------------------------------------
VP_COLOURS = [(230, 57, 70), (29, 120, 220), (46, 170, 90)]


def draw_evidence(res: GeometryResult, show_vp_rays: bool = True) -> np.ndarray:
    """Overlay: segments coloured by VP, thickness/alpha by convergence error;
    unexplained long lines in yellow; short rays towards each finite VP."""
    img = res.image_rgb.copy()
    over = img.copy()
    h, w = img.shape[:2]
    diag = np.hypot(h, w)
    for (x1, y1, x2, y2), lab, r in zip(res.segments, res.labels, res.residuals):
        p, q = (int(x1), int(y1)), (int(x2), int(y2))
        if lab >= 0:
            col = VP_COLOURS[lab % 3]
            cv2.line(over, p, q, col, 2, cv2.LINE_AA)
        elif np.hypot(x2 - x1, y2 - y1) >= LONG_LEN_FRAC * diag:
            cv2.line(over, p, q, (255, 200, 0), 2, cv2.LINE_AA)
    if show_vp_rays:
        for k, v in enumerate(res.vps_px):
            if abs(v[2]) < 1e-9:
                continue
            vx, vy = v[0] / v[2], v[1] / v[2]
            sel = np.flatnonzero(res.labels == k)
            for s in sel[np.argsort(-np.hypot(res.segments[sel, 2] - res.segments[sel, 0],
                                             res.segments[sel, 3] - res.segments[sel, 1]))[:12]]:
                mx, my = (res.segments[s, :2] + res.segments[s, 2:]) / 2
                # extend the segment's midpoint toward the VP (clipped to a long ray)
                dx, dy = vx - mx, vy - my
                t = min(1.0, 0.6 * diag / (np.hypot(dx, dy) + 1e-9))
                cv2.line(over, (int(mx), int(my)), (int(mx + t * dx), int(my + t * dy)),
                         VP_COLOURS[k % 3], 1, cv2.LINE_AA)
            if 0 <= vx < w and 0 <= vy < h:
                cv2.circle(over, (int(vx), int(vy)), 6, VP_COLOURS[k % 3], -1, cv2.LINE_AA)
    return cv2.addWeighted(over, 0.85, img, 0.15, 0)

In [ ]:
%%writefile {OUT_DIR}/physfeat/lighting.py
"""
Lighting / shading physics features for AI-generated image detection.

Idea
----
Under a distant light, the (low-frequency) brightness of a Lambertian surface is
    I(x) ≈ a + l · n(x)            (first-order spherical harmonics)
where n(x) is the surface normal and l points toward the light.  In a real
photograph ONE lighting environment explains the shading of every surface, so a
lighting model fitted on one part of the image predicts the others, and local
light-direction estimates agree.  Generators paint shading that is locally
plausible but globally inconsistent.

Given an RGB image and a per-pixel normal map (from a monocular normal estimator,
camera frame: x right, y up, z toward the viewer), we measure:
  * how well one global lighting model explains the shading (R²),
  * whether locally estimated light directions agree (grid of cells),
  * whether a model fitted on the other cells predicts each cell (leave-one-out R²),
  * whether specular highlights sit where the half-vector says they should,
  * whether the normal field itself is physically plausible (integrability).

All fits run on a small (≤128 px) version of the maps; ~5–10 ms per image.
"""
from __future__ import annotations

import cv2
import numpy as np

WORK_SIDE = 128           # maps are resized to this (longer side) before fitting
GRID = 4                  # GRID x GRID cells for local lighting estimates
MIN_CELL_PIX = 120        # a cell needs this many valid pixels ...
MIN_NORMAL_SPREAD = 0.02  # ... and this much normal variation (2nd eigenvalue) to estimate light
LUM_LO, LUM_HI = 0.03, 0.97
SHADING_BLUR = 1.5        # px; suppresses fine albedo texture, keeps low-frequency shading
VIEW = np.array([0.0, 0.0, 1.0])

FEATURE_NAMES = [
    "lit_r2_sh1",          # R² of global first-order lighting model
    "lit_r2_sh2",          # R² of global second-order (9-coeff SH) model
    "lit_r2_gain",         # sh2 - sh1: how much "complex lighting" is needed
    "lit_strength",        # |l| / (|a| + |l|): directional vs ambient
    "lit_elev",            # z-component of global light direction
    "lit_n_cells",         # number of cells with a usable local estimate
    "lit_cell_ang_med",    # median angle (deg) between local and global light directions
    "lit_cell_ang_p90",    # 90th percentile of that angle
    "lit_cell_resultant",  # mean resultant length of local light directions (1 = all agree)
    "lit_loo_r2",          # leave-one-cell-out R²: global model fitted without a cell predicts it
    "lit_loo_gap",         # in-cell R² minus leave-one-out R² (overfit gap = local inconsistency)
    "nrm_curl",            # normalised curl of the gradient field implied by normals (0 = integrable)
    "nrm_smooth",          # median angular change between neighbouring normals (deg)
    "nrm_frontal",         # mean n_z (how much of the scene faces the camera)
    "spec_frac",           # fraction of pixels that look like specular highlights
    "spec_align",          # mean n·h over highlights minus mean n·h over all pixels
]


# ----------------------------------------------------------------------------
# helpers
# ----------------------------------------------------------------------------
def _srgb_to_linear(x):
    return np.where(x <= 0.04045, x / 12.92, ((x + 0.055) / 1.055) ** 2.4)


def _resize_pair(rgb, normals, side=WORK_SIDE):
    h, w = normals.shape[:2]
    s = side / max(h, w)
    size = (max(8, round(w * s)), max(8, round(h * s)))
    rgb_s = cv2.resize(rgb, size, interpolation=cv2.INTER_AREA)
    n_s = cv2.resize(normals.astype(np.float32), size, interpolation=cv2.INTER_AREA)
    n_s /= np.linalg.norm(n_s, axis=-1, keepdims=True) + 1e-8
    return rgb_s, n_s


def sh1_basis(n):
    return np.concatenate([np.ones((len(n), 1)), n], 1)                       # 4


def sh2_basis(n):
    x, y, z = n[:, 0], n[:, 1], n[:, 2]
    return np.stack([np.ones_like(x), x, y, z, x * y, x * z, y * z,
                     x * x - y * y, 3 * z * z - 1], 1)                        # 9


def _fit(A, b, iters=3, c=2.0):
    """Huber IRLS least squares. Returns coefficients and weights."""
    w = np.ones(len(b))
    coef = np.zeros(A.shape[1])
    for _ in range(iters):
        sw = np.sqrt(w)[:, None]
        coef, *_ = np.linalg.lstsq(A * sw, b * sw[:, 0], rcond=None)
        r = b - A @ coef
        s = 1.4826 * np.median(np.abs(r - np.median(r))) + 1e-6
        u = np.abs(r) / (c * s)
        w = 1.0 / np.maximum(u, 1.0)
    return coef, w


def _r2(A, b, coef, w=None):
    r = b - A @ coef
    w = np.ones(len(b)) if w is None else w
    mu = np.sum(w * b) / w.sum()
    return float(1 - np.sum(w * r * r) / (np.sum(w * (b - mu) ** 2) + 1e-12))


def _unit(v):
    return v / (np.linalg.norm(v) + 1e-12)


def _ang(a, b):
    return float(np.degrees(np.arccos(np.clip(np.dot(_unit(a), _unit(b)), -1, 1))))


# ----------------------------------------------------------------------------
# main entry point
# ----------------------------------------------------------------------------
def extract(rgb: np.ndarray, normals: np.ndarray, return_debug: bool = False):
    """rgb: HxWx3 uint8; normals: hxwx3 float in [-1,1] (any resolution, same aspect)."""
    f = {k: np.nan for k in FEATURE_NAMES}
    rgb_s, n = _resize_pair(rgb, normals)
    h, w = n.shape[:2]
    lin = _srgb_to_linear(rgb_s.astype(np.float32) / 255.0)
    lum = 0.2126 * lin[..., 0] + 0.7152 * lin[..., 1] + 0.0722 * lin[..., 2]
    shade = cv2.GaussianBlur(lum, (0, 0), SHADING_BLUR)
    chroma = rgb_s.max(-1).astype(np.float32) / 255 - rgb_s.min(-1).astype(np.float32) / 255

    valid = (lum > LUM_LO) & (lum < LUM_HI) & np.isfinite(n).all(-1)
    # ---- normal-field plausibility -------------------------------------------------
    nz = np.clip(n[..., 2], 0.15, None)
    p, q = -n[..., 0] / nz, n[..., 1] / nz          # dz/dx, dz/dy (image y points down, normal y up)
    dp_dy = np.diff(p, axis=0)[:, :-1]
    dq_dx = np.diff(q, axis=1)[:-1, :]
    grad_mag = np.abs(np.diff(p, axis=1)[:-1, :]) + np.abs(np.diff(q, axis=0)[:, :-1])
    f["nrm_curl"] = float(np.median(np.abs(dp_dy - dq_dx)) / (np.median(grad_mag) + 1e-6))
    dots = np.concatenate([(n[1:] * n[:-1]).sum(-1).ravel(), (n[:, 1:] * n[:, :-1]).sum(-1).ravel()])
    f["nrm_smooth"] = float(np.degrees(np.median(np.arccos(np.clip(dots, -1, 1)))))
    f["nrm_frontal"] = float(n[..., 2].mean())

    if valid.sum() < 200:
        return (f, None) if return_debug else f

    N, B = n[valid], shade[valid]
    # ---- global lighting -------------------------------------------------------------
    A1, A2 = sh1_basis(N), sh2_basis(N)
    c1, w1 = _fit(A1, B)
    c2, w2 = _fit(A2, B)
    f["lit_r2_sh1"] = _r2(A1, B, c1, w1)
    f["lit_r2_sh2"] = _r2(A2, B, c2, w2)
    f["lit_r2_gain"] = f["lit_r2_sh2"] - f["lit_r2_sh1"]
    l_glob = c1[1:]
    f["lit_strength"] = float(np.linalg.norm(l_glob) / (abs(c1[0]) + np.linalg.norm(l_glob) + 1e-9))
    f["lit_elev"] = float(_unit(l_glob)[2])

    # ---- local (cell) lighting ------------------------------------------------------
    ys = np.linspace(0, h, GRID + 1).astype(int)
    xs = np.linspace(0, w, GRID + 1).astype(int)
    cell_id = np.full((h, w), -1, int)
    for i in range(GRID):
        for j in range(GRID):
            cell_id[ys[i]:ys[i + 1], xs[j]:xs[j + 1]] = i * GRID + j
    cid = cell_id[valid]
    # weighted normal equations per cell (global Huber weights) -> O(1) leave-one-out fits
    Aw = A1 * w1[:, None]
    G_all, r_all = A1.T @ Aw, Aw.T @ B
    dirs, wts, angs, loo, gaps = [], [], [], [], []
    for k in range(GRID * GRID):
        m = cid == k
        if m.sum() < MIN_CELL_PIX:
            continue
        ev = np.linalg.eigvalsh(np.cov(N[m].T))
        if ev[-2] < MIN_NORMAL_SPREAD:          # normals too uniform to pin down a light direction
            continue
        ck, wk = _fit(A1[m], B[m])
        r2_in = _r2(A1[m], B[m], ck, wk)
        if not np.isfinite(r2_in) or np.linalg.norm(ck[1:]) < 1e-4:
            continue
        # leave-one-cell-out: fit the lighting on every other pixel, predict this cell
        G_k, r_k = A1[m].T @ Aw[m], Aw[m].T @ B[m]
        co = np.linalg.solve(G_all - G_k + 1e-6 * np.eye(4), r_all - r_k)
        # allow a per-cell albedo gain/offset (albedo differs between regions, lighting does not)
        pred = A1[m] @ co
        G = np.c_[pred, np.ones(m.sum())]
        g, *_ = np.linalg.lstsq(G, B[m], rcond=None)
        r2_out = _r2(G, B[m], g)
        dirs.append(_unit(ck[1:]))
        wts.append(max(r2_in, 0) * m.sum())
        angs.append(_ang(ck[1:], l_glob))
        loo.append(r2_out)
        gaps.append(r2_in - r2_out)
    f["lit_n_cells"] = float(len(dirs))
    if len(dirs) >= 2:
        wts = np.asarray(wts) + 1e-9
        f["lit_cell_ang_med"] = float(np.median(angs))
        f["lit_cell_ang_p90"] = float(np.percentile(angs, 90))
        f["lit_cell_resultant"] = float(np.linalg.norm((np.asarray(dirs) * wts[:, None]).sum(0)) / wts.sum())
        f["lit_loo_r2"] = float(np.average(loo, weights=wts))
        f["lit_loo_gap"] = float(np.average(gaps, weights=wts))

    # ---- specular highlights -------------------------------------------------------
    spec = valid & (lum > np.percentile(lum[valid], 99)) & (chroma < 0.12) & (lum > 0.5)
    f["spec_frac"] = float(spec.mean())
    if spec.sum() >= 5:
        hvec = _unit(_unit(l_glob) + VIEW)
        ndh = n @ hvec
        f["spec_align"] = float(ndh[spec].mean() - ndh[valid].mean())

    if return_debug:
        pred = np.full((h, w), np.nan, np.float32)
        pred[valid] = A1 @ c1
        dbg = {"rgb": rgb_s, "normals": n, "shade": shade, "pred": pred, "valid": valid,
               "light": _unit(l_glob), "cell_id": cell_id}
        return f, dbg
    return f


def draw_evidence(dbg) -> np.ndarray:
    """3-panel strip: normals | observed shading | shading predicted by ONE global light
    (with the estimated light direction drawn as an arrow)."""
    n_vis = ((dbg["normals"] * [1, 1, 1] + 1) * 127.5).clip(0, 255).astype(np.uint8)
    def gray(x):
        v = np.nan_to_num(x, nan=0.0)
        lo, hi = np.percentile(v[dbg["valid"]], [1, 99]) if dbg["valid"].any() else (0, 1)
        return (np.clip((v - lo) / (hi - lo + 1e-6), 0, 1) * 255).astype(np.uint8)
    obs = cv2.cvtColor(gray(dbg["shade"]), cv2.COLOR_GRAY2RGB)
    pred = cv2.cvtColor(gray(dbg["pred"]), cv2.COLOR_GRAY2RGB)
    h, w = pred.shape[:2]
    l = dbg["light"]
    c = (w // 2, h // 2)
    tip = (int(c[0] + l[0] * 0.4 * w), int(c[1] - l[1] * 0.4 * h))   # y up -> image y down
    cv2.arrowedLine(pred, c, tip, (255, 180, 0), 2, tipLength=0.25)
    return np.concatenate([n_vis, obs, pred], 1)

In [ ]:
%%writefile {OUT_DIR}/physfeat/normals.py
"""Monocular surface-normal estimation (GPU) + the lighting features that consume it.

Backends
--------
* "marigold" (default): prs-eth/marigold-normals-v1-1 via diffusers.MarigoldNormalsPipeline.
  Output: unit normals in the camera frame, x right, y up, z toward the viewer.
* "depth": depth-anything/Depth-Anything-V2-Small-hf via transformers; normals are
  derived from the gradients of the relative (inverse) depth. Much faster, coarser.
  Used automatically if Marigold cannot be loaded.

Every image is centre-cropped to a square and resized to RES x RES so batches have
one shape; the lighting features are computed on exactly the same crop.
"""
from __future__ import annotations

import gc

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from . import lighting
from .imaging import load_pil

MARIGOLD_ID = "prs-eth/marigold-normals-v1-1"
DEPTH_ID = "depth-anything/Depth-Anything-V2-Small-hf"


def square_crop(im, res):
    w, h = im.size
    s = min(w, h)
    left, top = (w - s) // 2, (h - s) // 2
    return im.crop((left, top, left + s, top + s)).resize((res, res), resample=3)  # bicubic


class CropList(Dataset):
    def __init__(self, paths, res, normalise_format=True, perturb=None):
        self.paths, self.res, self.norm, self.perturb = list(paths), res, normalise_format, perturb

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            im = square_crop(load_pil(self.paths[i], self.norm, self.perturb), self.res)
            return torch.from_numpy(np.asarray(im).copy()), True
        except Exception:
            return torch.zeros(self.res, self.res, 3, dtype=torch.uint8), False


class NormalEstimator:
    def __init__(self, backend="marigold", device="cuda", steps=None, res=512, model_id=None):
        self.device, self.steps, self.res = device, steps, res
        self.dtype = torch.float16 if device.startswith("cuda") else torch.float32
        self.backend = backend
        if backend == "marigold":
            from diffusers import MarigoldNormalsPipeline
            mid = model_id or MARIGOLD_ID
            try:
                self.pipe = MarigoldNormalsPipeline.from_pretrained(mid, variant="fp16", torch_dtype=self.dtype)
            except Exception:
                self.pipe = MarigoldNormalsPipeline.from_pretrained(mid, torch_dtype=self.dtype)
            self.pipe = self.pipe.to(device)
            self.pipe.set_progress_bar_config(disable=True)
        elif backend == "depth":
            from transformers import AutoModelForDepthEstimation
            self.model = AutoModelForDepthEstimation.from_pretrained(
                model_id or DEPTH_ID, torch_dtype=self.dtype).to(device).eval()
        else:
            raise ValueError(backend)

    @torch.no_grad()
    def __call__(self, batch_uint8: torch.Tensor) -> np.ndarray:
        """batch_uint8: (B,R,R,3) uint8 -> (B,R',R',3) float32 unit normals (x right, y up, z to viewer)."""
        x = batch_uint8.permute(0, 3, 1, 2).float().div(255.0)
        if self.backend == "marigold":
            out = self.pipe(x.to(self.device, self.dtype), num_inference_steps=self.steps,
                            processing_resolution=self.res, match_input_resolution=False,
                            batch_size=len(x), output_type="np",
                            generator=torch.Generator(device=self.device).manual_seed(0))
            return np.asarray(out.prediction, np.float32)
        # ---- depth backend: normals from relative inverse-depth gradients ----------------
        side = (self.res // 14) * 14
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        inp = F.interpolate((x - mean) / std, size=(side, side), mode="bilinear", align_corners=False)
        d = self.model(pixel_values=inp.to(self.device, self.dtype)).predicted_depth.float()
        d = F.interpolate(d[:, None], size=(side, side), mode="bilinear", align_corners=False)[:, 0]
        d = d / (d.flatten(1).quantile(0.95, dim=1).view(-1, 1, 1) + 1e-6)   # scale-normalise
        gx = torch.zeros_like(d); gy = torch.zeros_like(d)
        gx[:, :, 1:-1] = (d[:, :, 2:] - d[:, :, :-2]) / 2
        gy[:, 1:-1, :] = (d[:, 2:, :] - d[:, :-2, :]) / 2
        k = 0.577 / (torch.sqrt(gx ** 2 + gy ** 2).flatten(1).median(1).values.view(-1, 1, 1) + 1e-6)
        n = torch.stack([-gx * k, gy * k, torch.ones_like(d)], -1)   # height = disparity (toward viewer)
        n = n / n.norm(dim=-1, keepdim=True)
        return n.cpu().numpy().astype(np.float32)


def lighting_features(estimator: NormalEstimator, paths, batch_size=8, num_workers=4,
                      normalise_format=True, perturb=None, progress=True, keep_examples=0):
    """Returns (list of feature dicts, list of (rgb, normals) examples for plotting)."""
    dl = DataLoader(CropList(paths, estimator.res, normalise_format, perturb), batch_size=batch_size,
                    num_workers=num_workers, pin_memory=estimator.device.startswith("cuda"))
    feats, examples = [], []
    it = dl
    bar = None
    if progress:
        from tqdm.auto import tqdm
        bar = it = tqdm(dl, desc=f"normals{'/' + perturb if perturb else ''}", leave=False)
    try:
        for imgs, ok in it:
            normals = estimator(imgs)
            rgbs = imgs.numpy()
            for rgb, n, good in zip(rgbs, normals, ok.numpy()):
                if not good:
                    f = {k: np.nan for k in lighting.FEATURE_NAMES}
                else:
                    try:
                        f = lighting.extract(rgb, n)
                    except Exception:
                        f = {k: np.nan for k in lighting.FEATURE_NAMES}
                feats.append(f)
                if len(examples) < keep_examples and good:
                    examples.append((rgb, n.astype(np.float16)))
    finally:  # tidy DataLoader workers explicitly (avoids noisy __del__ tracebacks in forked children)
        if bar is not None:
            bar.close()
        del it, dl
        gc.collect()
    return feats, examples

In [ ]:
%%writefile {OUT_DIR}/physfeat/dataset.py
"""Find GenImage-style images on disk and build a labelled manifest.

Works with the original GenImage layout
    imagenet_ai_0419_sdv4/train/ai/*.png, .../val/nature/*.JPEG
and with the common Kaggle re-uploads (tiny-genimage, unbiased-tiny-genimage),
which use folder names like  midjourney/ai, sd14/nature, real/, fake/ ...
Labels, generator and split are inferred from path components.
"""
from __future__ import annotations

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

from .imaging import IMG_EXTS

REAL_TOKENS = {"nature", "real", "0real", "natural", "imagenet", "authentic", "genuine", "0"}
FAKE_TOKENS = {"ai", "fake", "1fake", "generated", "synthetic", "aigenerated", "1"}
SPLIT_TOKENS = {"train": "train", "training": "train", "val": "val", "valid": "val",
                "validation": "val", "test": "test", "testing": "test"}

# normalised alias -> canonical generator name (checked as substrings, longest first)
GEN_ALIASES = {
    "midjourney": "midjourney", "imagenetmidjourney": "midjourney", "mj": "midjourney",
    "stablediffusionv14": "sd14", "stablediffusionv4": "sd14", "sdv14": "sd14", "sd14": "sd14",
    "sdv4": "sd14",
    "stablediffusionv15": "sd15", "stablediffusionv5": "sd15", "sdv15": "sd15", "sd15": "sd15",
    "sdv5": "sd15",
    "adm": "adm", "glide": "glide", "wukong": "wukong", "vqdm": "vqdm", "biggan": "biggan",
    "sdxl": "sdxl", "flux": "flux", "dalle": "dalle",
    "stablediffusion": "sd", "sd": "sd",   # version-less fallbacks (checked after the specific ones)
}
_ALIAS_ORDER = sorted(GEN_ALIASES, key=len, reverse=True)
_SHORT = {"mj", "adm", "sd"}  # short aliases must match a whole component


def _norm(s: str) -> str:
    return re.sub(r"[^a-z0-9]", "", s.lower())


def parse_path(path: str) -> dict:
    parts = [_norm(p) for p in Path(path).parts[:-1]]
    label, gen, split = None, None, None
    for p in reversed(parts):               # innermost folder wins
        if label is None and p in REAL_TOKENS:
            label = 0
        elif label is None and p in FAKE_TOKENS:
            label = 1
        if split is None and p in SPLIT_TOKENS:
            split = SPLIT_TOKENS[p]
        if gen is None:
            for a in _ALIAS_ORDER:
                if (a in _SHORT and (p == a or p.endswith(a) and p.startswith("imagenet"))) or \
                   (a not in _SHORT and a in p):
                    gen = GEN_ALIASES[a]
                    break
    if label is None and gen is not None:
        # flat layouts (e.g. unbiased-tiny-genimage: BigGAN/, ADM/, ..., plus a real folder):
        # a generator-named folder with no real/fake sub-folder holds that generator's fakes
        label = 1
    return {"label": label, "subset": gen, "split": split}


def scan(roots, exts=IMG_EXTS) -> pd.DataFrame:
    rows = []
    for root in roots:
        for dp, _, fns in os.walk(root, followlinks=True):
            for fn in fns:
                if os.path.splitext(fn)[1].lower() in exts:
                    p = os.path.join(dp, fn)
                    rows.append({"path": p, **parse_path(os.path.relpath(p, root))})
    df = pd.DataFrame(rows, columns=["path", "label", "subset", "split"])
    # fakes are attributed to the generator of their subset; reals are just "real"
    df["generator"] = np.where(df["label"] == 0, "real", df["subset"])
    return df


def folder_tree(root, max_depth=4, max_children=12) -> str:
    """Compact tree with image counts, to show what the scanner is looking at."""
    lines = []

    def count(d):
        n = 0
        for _, _, fns in os.walk(d):
            n += sum(os.path.splitext(f)[1].lower() in IMG_EXTS for f in fns)
        return n

    def rec(d, depth):
        subs = sorted([e for e in os.scandir(d) if e.is_dir()], key=lambda e: e.name)
        for e in subs[:max_children]:
            lines.append("  " * depth + f"{e.name}/  ({count(e.path)} imgs)")
            if depth + 1 < max_depth:
                rec(e.path, depth + 1)
        if len(subs) > max_children:
            lines.append("  " * depth + f"... {len(subs) - max_children} more")
    lines.append(f"{root}/  ({count(root)} imgs)")
    rec(root, 1)
    return "\n".join(lines)

In [ ]:
%%writefile {OUT_DIR}/physfeat/protocol.py
"""Cross-generator evaluation protocol.

Train:  real images (train split)  vs  fakes from ONE generator (train split)
Test:   for every generator g:  real images (val split)  vs  fakes from g (val split)

The generator used for training is "seen"; every other generator is "unseen".
Generalisation = performance on the unseen ones.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

PREFERRED_TRAIN_GENERATORS = ["sd14", "sd15", "sd", "adm", "glide", "vqdm", "wukong", "midjourney", "biggan"]


def pick_train_generator(df: pd.DataFrame, requested: str = "auto") -> str:
    gens = df.loc[(df.label == 1) & (df.split == "train"), "generator"].value_counts()
    if gens.empty:
        raise ValueError("No fake images in the train split — the folder scanner labelled nothing as fake. "
                         "Check the folder tree printed above and extend GEN_ALIASES / FAKE_TOKENS "
                         "in physfeat/dataset.py. Counts per generator/split:\n"
                         + df.groupby(["generator", "split"]).size().to_string())
    if requested != "auto":
        if requested not in gens.index:
            raise ValueError(f"{requested!r} has no training fakes; available: {list(gens.index)}")
        return requested
    for g in PREFERRED_TRAIN_GENERATORS:
        if gens.get(g, 0) >= 200:
            return g
    return gens.index[0]


def train_mask(df, train_gen):
    return (df.split == "train").values & ((df.label == 0) | (df.generator == train_gen)).values


def test_masks(df, min_fakes=5):
    """dict generator -> boolean mask over df (val reals + val fakes of that generator)."""
    val = (df.split == "val").values
    real = val & (df.label == 0).values
    out = {}
    for g in sorted(df.loc[df.label == 1, "generator"].dropna().unique()):
        m = real | (val & (df.generator == g).values)
        if (m & (df.label == 1).values).sum() >= min_fakes:
            out[g] = m
    return out


def evaluate(scores: dict, df: pd.DataFrame, train_gen: str, extra_mask=None) -> pd.DataFrame:
    """scores: model name -> array of P(fake) over the whole df. Returns a tidy table."""
    rows = []
    for g, m in test_masks(df).items():
        if extra_mask is not None:
            m = m & extra_mask
        y = df.label.values[m]
        if len(np.unique(y)) < 2:
            continue
        for name, s in scores.items():
            p = s[m]
            rows.append({"generator": g, "seen": g == train_gen, "model": name,
                         "auc": roc_auc_score(y, p),
                         "bal_acc": balanced_accuracy_score(y, p > 0.5),
                         "n_fake": int(y.sum()), "n_real": int((1 - y).sum())})
    return pd.DataFrame(rows)


def summary_table(res: pd.DataFrame, metric="auc") -> pd.DataFrame:
    if res.empty:
        return pd.DataFrame({"note": ["no generator had enough validation fakes to evaluate"]})
    t = res.pivot_table(index=["generator", "seen"], columns="model", values=metric).reset_index()
    t = t.sort_values(["seen", "generator"], ascending=[False, True]).set_index("generator")
    unseen = t[~t.seen].drop(columns="seen")
    t = t.drop(columns="seen")
    t.loc["MEAN (unseen)"] = unseen.mean()
    return t


def bootstrap_mean_auc(scores: dict, df: pd.DataFrame, train_gen: str, pairs, n_boot=500, seed=0):
    """Paired bootstrap of the mean AUC over unseen generators.

    Resamples real and fake test images (stratified by generator, reals shared across
    generators exactly as in `evaluate`) and recomputes every model's mean unseen AUC on
    the same resample. Returns per-model [mean, lo, hi] and per-pair difference [mean, lo, hi, P(diff<=0)].
    """
    rng = np.random.default_rng(seed)
    val = (df.split == "val").values
    real_idx = np.flatnonzero(val & (df.label == 0).values)
    gens = [g for g in test_masks(df) if g != train_gen]
    fake_idx = {g: np.flatnonzero(val & (df.generator == g).values) for g in gens}
    names = list(scores)
    S = {k: np.asarray(v) for k, v in scores.items()}

    def mean_auc(ri, fi):
        out = {k: [] for k in names}
        for g in gens:
            idx = np.r_[ri, fi[g]]
            yy = np.r_[np.zeros(len(ri)), np.ones(len(fi[g]))]
            for k in names:
                out[k].append(roc_auc_score(yy, S[k][idx]))
        return {k: float(np.mean(v)) for k, v in out.items()}

    point = mean_auc(real_idx, fake_idx)
    boots = []
    for _ in range(n_boot):
        ri = rng.choice(real_idx, len(real_idx))
        fi = {g: rng.choice(v, len(v)) for g, v in fake_idx.items()}
        boots.append(mean_auc(ri, fi))
    B = pd.DataFrame(boots)
    models = {k: [point[k], float(B[k].quantile(.025)), float(B[k].quantile(.975))] for k in names}
    diffs = {}
    for a, b in pairs:
        d = B[a] - B[b]
        diffs[f"{a}  minus  {b}"] = [point[a] - point[b], float(d.quantile(.025)), float(d.quantile(.975)),
                                    float((d <= 0).mean())]
    return models, diffs

In [ ]:
%%writefile {OUT_DIR}/physfeat/workers.py
"""Picklable per-image workers for multiprocessing pools."""
from __future__ import annotations

import numpy as np

from . import geometry
from .imaging import load_rgb, read_meta, stable_seed


def meta_worker(path):
    return read_meta(path)


def geometry_worker(args):
    """args = (path, normalise_format, perturb). Never raises."""
    path, normalise, perturb = args
    try:
        rgb = load_rgb(path, normalise, perturb)
        feats = geometry.extract(rgb, seed=stable_seed(path)).features
        feats["geo_error"] = None
    except Exception as e:
        feats = {k: np.nan for k in geometry.FEATURE_NAMES}
        feats["geo_error"] = repr(e)[:200]
    return feats

In [ ]:
%%writefile {OUT_DIR}/physfeat/clipfeat.py
"""CLIP ViT-L/14 image embeddings (the UnivFD-style data-driven baseline)."""
from __future__ import annotations

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms as T

from .imaging import load_pil

CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)

PREPROCESS = T.Compose([
    T.Resize(224, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(CLIP_MEAN, CLIP_STD),
])


class ImageList(Dataset):
    def __init__(self, paths, normalise_format=True, perturb=None):
        self.paths, self.norm, self.perturb = list(paths), normalise_format, perturb

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            return PREPROCESS(load_pil(self.paths[i], self.norm, self.perturb)), True
        except Exception:
            return torch.zeros(3, 224, 224), False


def load_clip(model_name: str, device: str):
    from transformers import CLIPVisionModelWithProjection
    dtype = torch.float16 if device.startswith("cuda") else torch.float32
    model = CLIPVisionModelWithProjection.from_pretrained(model_name, torch_dtype=dtype)
    return model.to(device).eval()


@torch.no_grad()
def embed(model, paths, device, batch_size=128, num_workers=4,
          normalise_format=True, perturb=None, progress=True):
    """Returns (N, D) float16 L2-normalised embeddings and an (N,) ok-mask."""
    dl = DataLoader(ImageList(paths, normalise_format, perturb), batch_size=batch_size,
                    num_workers=num_workers, pin_memory=device.startswith("cuda"))
    dtype = next(model.parameters()).dtype
    feats, oks = [], []
    it, bar = dl, None
    if progress:
        from tqdm.auto import tqdm
        it = bar = tqdm(dl, desc=f"CLIP{'/' + perturb if perturb else ''}", leave=False)
    try:
        for x, ok in it:
            e = model(pixel_values=x.to(device, dtype)).image_embeds.float()
            e = torch.nn.functional.normalize(e, dim=-1)
            feats.append(e.cpu().numpy().astype(np.float16))
            oks.append(ok.numpy())
    finally:  # shut DataLoader workers down here, not in a later forked child (noisy tracebacks)
        if bar is not None:
            bar.close()
        del it, dl
        import gc
        gc.collect()
    return np.concatenate(feats), np.concatenate(oks)

In [ ]:
%%writefile {OUT_DIR}/physfeat/cache.py
"""Chunked, resumable feature computation (a chunk is reused only if its path list matches)."""
from __future__ import annotations

import time
from pathlib import Path

import numpy as np
import pandas as pd


def chunked(paths, name, fn, ext, feat_dir, chunk=2000, log=print):
    feat_dir = Path(feat_dir)
    feat_dir.mkdir(parents=True, exist_ok=True)
    paths = list(paths)
    outs = []
    for ci, s in enumerate(range(0, len(paths), chunk)):
        ps = paths[s:s + chunk]
        f, fp = feat_dir / f"{name}_{ci:04d}.{ext}", feat_dir / f"{name}_{ci:04d}.paths.txt"
        if f.exists() and fp.exists() and fp.read_text().splitlines() == ps:
            outs.append(pd.read_parquet(f) if ext == "parquet" else np.load(f))
            continue
        t0 = time.time()
        out = fn(ps)
        if ext == "parquet":
            out.to_parquet(f)
        else:
            np.save(f, out)
        fp.write_text("\n".join(ps))
        outs.append(out)
        log(f"{name} chunk {ci}: {len(ps)} imgs in {time.time() - t0:.0f}s")
    if not outs:
        return None
    return pd.concat(outs, ignore_index=True) if ext == "parquet" else np.concatenate(outs)

In [ ]:
%%writefile {OUT_DIR}/physfeat/models.py
"""Detectors: CLIP linear probe (baseline), physics GBMs, and stacked fusions."""
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold, cross_val_predict

from . import geometry, lighting

GROUPS = {
    "geometry": list(geometry.FEATURE_NAMES),
    "lighting": list(lighting.FEATURE_NAMES),
    "geometry+lighting": list(geometry.FEATURE_NAMES) + list(lighting.FEATURE_NAMES),
}
BASELINE = "CLIP (baseline)"


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def _gbm(seed):
    return HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, max_leaf_nodes=31,
                                          l2_regularization=1.0, class_weight="balanced",
                                          early_stopping=False, random_state=seed)


class Detector:
    def __init__(self, seed=0, clip_Cs=(0.25, 1.0, 4.0, 16.0), groups=None):
        self.seed, self.clip_Cs = seed, list(clip_Cs)
        self.groups = groups or GROUPS

    def fit(self, clip: np.ndarray, phys: pd.DataFrame, y: np.ndarray, log=print):
        cv = StratifiedKFold(5, shuffle=True, random_state=self.seed)
        sel = LogisticRegressionCV(Cs=self.clip_Cs, cv=3, scoring="roc_auc", class_weight="balanced",
                                   max_iter=4000).fit(clip, y)
        self.C = float(sel.C_[0])
        log(f"CLIP linear probe: C = {self.C}")
        self.clip = LogisticRegression(C=self.C, class_weight="balanced", max_iter=4000)
        oof = {"clip": cross_val_predict(self.clip, clip, y, cv=cv, method="predict_proba")[:, 1]}
        self.clip.fit(clip, y)
        self.phys = {}
        for g, cols in self.groups.items():
            m = _gbm(self.seed)
            oof[g] = cross_val_predict(m, phys[cols].values, y, cv=cv, method="predict_proba")[:, 1]
            self.phys[g] = m.fit(phys[cols].values, y)
        self.stack = {}
        for g in self.groups:
            Z = np.c_[logit(oof["clip"]), logit(oof[g])]
            self.stack[g] = LogisticRegression(class_weight="balanced").fit(Z, y)
            log(f"stack CLIP+{g}: weights {np.round(self.stack[g].coef_[0], 3)}")
        return self

    def predict(self, clip: np.ndarray, phys: pd.DataFrame) -> dict:
        s_clip = self.clip.predict_proba(clip)[:, 1]
        out = {BASELINE: s_clip}
        for g, cols in self.groups.items():
            s_g = self.phys[g].predict_proba(phys[cols].values)[:, 1]
            out[f"Physics: {g}"] = s_g
            out[f"Fused: CLIP+{g}"] = self.stack[g].predict_proba(np.c_[logit(s_clip), logit(s_g)])[:, 1]
        return out


# ----------------------------------------------------------------------------------------
# Compression-aware detector (notebook 03)
# ----------------------------------------------------------------------------------------
def quality_term(q):
    """0 for a pristine image (q=100), 1 at JPEG quality 50; NaN -> treated as pristine-ish 96."""
    q = np.where(np.isfinite(q), q, 96.0)
    return np.clip((100.0 - q) / 50.0, 0.0, 1.5)


def _stack_design(lc, lp, q, quality_aware):
    if not quality_aware:
        return np.c_[lc, lp]
    t = quality_term(q)
    return np.c_[lc, lp, t, t * lc, t * lp]


class RobustDetector:
    """Trained on clean + JPEG-augmented copies of the training images.

    * CLIP probe and physics GBM see both copies (standard augmentation).
    * Fusion = logistic stacking on out-of-fold logits. The quality-aware variant also gets
      the image's JPEG quality and its interactions with each branch, so the weight it puts
      on CLIP vs physics can change with compression.
    Out-of-fold predictions use GroupKFold over source images, so the clean and the
    compressed copy of one image never sit on both sides of a split.
    """
    NAMES = {"clip": "CLIP (JPEG-aug)", "phys": "Physics (JPEG-aug)",
             "fused": "Fused (JPEG-aug)", "fused_q": "Fused (JPEG-aug, quality-aware)"}

    def __init__(self, phys_cols=None, seed=0, clip_Cs=(0.25, 1.0, 4.0, 16.0)):
        self.phys_cols = phys_cols or GROUPS["geometry+lighting"]
        self.seed, self.clip_Cs = seed, list(clip_Cs)

    def fit(self, clip, phys, y, q, groups, log=print):
        from sklearn.model_selection import GroupKFold
        splits = list(GroupKFold(n_splits=5).split(clip, y, groups))
        sel = LogisticRegressionCV(Cs=self.clip_Cs, cv=splits[:3], scoring="roc_auc",
                                   class_weight="balanced", max_iter=4000).fit(clip, y)
        self.C = float(sel.C_[0])
        log(f"CLIP linear probe (augmented): C = {self.C}")
        self.clip = LogisticRegression(C=self.C, class_weight="balanced", max_iter=4000)
        X = phys[self.phys_cols].values
        self.phys = _gbm(self.seed)
        oof_c = cross_val_predict(self.clip, clip, y, cv=splits, method="predict_proba")[:, 1]
        oof_p = cross_val_predict(self.phys, X, y, cv=splits, method="predict_proba")[:, 1]
        self.clip.fit(clip, y)
        self.phys.fit(X, y)
        lc, lp = logit(oof_c), logit(oof_p)
        self.stack = LogisticRegression(class_weight="balanced", max_iter=2000).fit(
            _stack_design(lc, lp, q, False), y)
        self.stack_q = LogisticRegression(class_weight="balanced", max_iter=2000).fit(
            _stack_design(lc, lp, q, True), y)
        log(f"stack (no quality): {np.round(self.stack.coef_[0], 3)}")
        log(f"stack (quality-aware) [clip, phys, t, t*clip, t*phys]: {np.round(self.stack_q.coef_[0], 3)}")
        return self

    def branch_weights(self, q):
        """Effective stacking weights on the CLIP and physics logits at JPEG quality q."""
        w = self.stack_q.coef_[0]
        t = quality_term(np.asarray(q, float))
        return w[0] + w[3] * t, w[1] + w[4] * t

    def predict(self, clip, phys, q) -> dict:
        s_c = self.clip.predict_proba(clip)[:, 1]
        s_p = self.phys.predict_proba(phys[self.phys_cols].values)[:, 1]
        lc, lp = logit(s_c), logit(s_p)
        return {self.NAMES["clip"]: s_c, self.NAMES["phys"]: s_p,
                self.NAMES["fused"]: self.stack.predict_proba(_stack_design(lc, lp, q, False))[:, 1],
                self.NAMES["fused_q"]: self.stack_q.predict_proba(_stack_design(lc, lp, q, True))[:, 1]}

In [ ]:
import importlib, gc, subprocess
try:
    import diffusers  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers"], check=False)
import numpy as np, pandas as pd, torch, joblib
from tqdm.auto import tqdm
from multiprocessing import Pool
import physfeat
from physfeat import geometry as G, lighting as Lg, normals as NM, protocol as P, workers as W, models as M
from physfeat import imaging as I
from physfeat.cache import chunked
for m in (I, G, Lg, NM, P, W, M):
    importlib.reload(m)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
if IN_KAGGLE and DEVICE == "cpu":
    raise RuntimeError("No GPU: sidebar -> Session options -> Accelerator -> GPU T4, then Save Version again.")

## 2 · Load notebook 01 + 02 outputs

In [ ]:
def walk_find(pred, max_depth=6):
    hits = []
    for dp, dns, fns in os.walk(INPUT_ROOT, followlinks=True):
        depth = dp.count(os.sep) - INPUT_ROOT.count(os.sep)
        if depth > max_depth or len(fns) > 500:
            dns[:] = []
            continue
        dns[:] = [d for d in dns if d not in ("physfeat", "__pycache__")]
        if pred(Path(dp), fns):
            hits.append(Path(dp))
    return hits

def show_inputs():
    for dp, dns, fns in os.walk(INPUT_ROOT, followlinks=True):
        depth = dp.count(os.sep) - INPUT_ROOT.count(os.sep)
        if depth <= 3 and len(fns) < 500:
            print("  " * depth + os.path.basename(dp) + "/", fns[:6])
        if depth >= 3 or len(fns) >= 500:
            dns[:] = []

nb01 = Path(NB01_DIR) if NB01_DIR else next(iter(sorted(walk_find(
    lambda p, f: "manifest.csv" in f and (p / "features" / "clip_vitl14.npy").exists()))), None)
nb02 = Path(NB02_DIR) if NB02_DIR else next(iter(sorted(walk_find(
    lambda p, f: "results.json" in f and (p / "models" / "detector.joblib").exists()))), None)
if nb01 is None or nb02 is None:
    show_inputs()
    raise FileNotFoundError(f"missing input: notebook 01 output found={nb01}, notebook 02 output found={nb02}. "
                            "Add both notebook outputs (latest successful versions) as inputs.")
print("notebook 01:", nb01, "\nnotebook 02:", nb02)

manifest = pd.read_csv(nb01 / "manifest.csv")
clip_clean = np.load(nb01 / "features" / "clip_vitl14.npy").astype(np.float32)
geo_clean = pd.read_parquet(nb01 / "features" / "geometry.parquet")
res02 = json.loads((nb02 / "results.json").read_text())
TRAIN_GEN = res02["train_generator"]
NORMAL_BACKEND, NORMAL_RES, NORMAL_STEPS = (res02["normals"][k] for k in ("backend", "res", "steps"))
TAG = f"{NORMAL_BACKEND}{NORMAL_RES}s{NORMAL_STEPS or 'd'}"
light_clean = pd.read_parquet(nb02 / "features" / f"lighting_{TAG}.parquet")
assert len(light_clean) == len(manifest)

# image paths must resolve here (dataset attached); remap the dataset root if the mount moved
missing = [p for p in manifest.path.head(50) if not os.path.exists(p)]
if missing:
    name = Path(missing[0]).name
    found = []
    for dp, dns, fns in os.walk(INPUT_ROOT, followlinks=True):   # follows Kaggle's symlinked mounts
        if name in fns:
            found.append(os.path.join(dp, name)); break
    if not found:
        print(f"Looking for e.g. {missing[0]}\nWhat is attached under {INPUT_ROOT}:")
        for dp, dns, fns in os.walk(INPUT_ROOT, followlinks=True):
            depth = dp.count(os.sep) - INPUT_ROOT.count(os.sep)
            print("  " * depth + os.path.basename(dp) + f"/  ({len(fns)} files)")
            if depth >= 3 or len(fns) >= 500:
                dns[:] = []
        raise FileNotFoundError("dataset images not found: Add Input -> Datasets -> 'Unbiased Tiny GenImage' "
                                "(cartografia/unbiased-tiny-genimage). It must be attached to THIS notebook too.")
    old_base = str(Path(missing[0]).parent.parent); new_base = str(Path(found[0]).parent.parent)
    manifest["path"] = manifest.path.str.replace(old_base, new_base, regex=False)
    print(f"remapped image paths: {old_base} -> {new_base}")

key = lambda p: "/".join(Path(p).parts[-2:])          # generator-folder/filename: stable across mounts
manifest["key"] = manifest.path.map(key)
assert manifest.key.is_unique
tr = P.train_mask(manifest, TRAIN_GEN)
va = (manifest.split == "val").values
val_idx, tr_idx = np.flatnonzero(va), np.flatnonzero(tr)
print(f"seen generator {TRAIN_GEN} · train {len(tr_idx):,} · val {len(val_idx):,} · normals {TAG}")

In [ ]:
def load_cached(prefix, ext):
    # concatenate notebook-02 chunk files <prefix>_NNNN.<ext> in order, with their path lists
    files = sorted(glob.glob(str(nb02 / "features" / f"{prefix}_[0-9][0-9][0-9][0-9].{ext}")))
    if not files:
        raise FileNotFoundError(f"notebook-02 cache missing: {prefix}_*.{ext}")
    paths, parts = [], []
    for f in files:
        paths += Path(f[: -len(ext) - 1] + ".paths.txt").read_text().splitlines()
        parts.append(pd.read_parquet(f) if ext == "parquet" else np.load(f))
    data = pd.concat(parts, ignore_index=True) if ext == "parquet" else np.concatenate(parts)
    pos = pd.Series(range(len(paths)), index=[key(p) for p in paths])
    order = pos.loc[manifest.key.values[val_idx]].values   # align to this notebook's val order
    return data.iloc[order].reset_index(drop=True) if ext == "parquet" else data[order]

PHYS_COLS = M.GROUPS["geometry+lighting"]
phys_clean = pd.concat([geo_clean[G.FEATURE_NAMES], light_clean[Lg.FEATURE_NAMES]], axis=1)

conds = {"clean": (clip_clean[val_idx], phys_clean.iloc[val_idx].reset_index(drop=True))}
for pert in PERTURBATIONS:
    c = load_cached(f"clip_{pert}", "npy").astype(np.float32)
    g = load_cached(f"geo_{pert}", "parquet")
    l = load_cached(f"light_{TAG}_{pert}", "parquet")
    conds[pert] = (c, pd.concat([g[G.FEATURE_NAMES], l[Lg.FEATURE_NAMES]], axis=1))
    print(f"loaded cached test features: {pert} {c.shape}")

## 3 · Features of the training images after random JPEG compression (q 50–95)

In [ ]:
from physfeat import clipfeat as C
tr_paths = manifest.path.values[tr_idx].tolist()
q_aug = np.array([I.jpegrand_quality(p) for p in tr_paths], float)
print(f"augmentation quality: min {q_aug.min():.0f}, mean {q_aug.mean():.1f}, max {q_aug.max():.0f}")

clip_model = C.load_clip(CLIP_MODEL, DEVICE)
clip_aug = chunked(tr_paths, "clip_jpegrand", lambda ps: C.embed(
    clip_model, ps, DEVICE, batch_size=128 if DEVICE == "cuda" else 16, num_workers=N_CPU,
    normalise_format=NORMALISE_FORMAT, perturb="jpegrand")[0], "npy", FEAT_DIR, CHUNK).astype(np.float32)
del clip_model; gc.collect()

def geo_fn(ps):
    with Pool(N_CPU) as pool:
        rows = list(tqdm(pool.imap(W.geometry_worker, [(p, NORMALISE_FORMAT, "jpegrand") for p in ps], chunksize=8),
                         total=len(ps), desc="geometry/jpegrand", leave=False))
    out = pd.DataFrame(rows); out["geo_error"] = out["geo_error"].astype("string")
    return out
geo_aug = chunked(tr_paths, "geo_jpegrand", geo_fn, "parquet", FEAT_DIR, CHUNK)

est = NM.NormalEstimator(NORMAL_BACKEND, DEVICE, steps=NORMAL_STEPS, res=NORMAL_RES, model_id=NORMAL_MODEL)
def light_fn(ps):
    feats, _ = NM.lighting_features(est, ps, batch_size=NORMAL_BATCH, num_workers=N_CPU,
                                    normalise_format=NORMALISE_FORMAT, perturb="jpegrand")
    return pd.DataFrame(feats, columns=Lg.FEATURE_NAMES)
t0 = time.time()
light_aug = chunked(tr_paths, f"light_{TAG}_jpegrand", light_fn, "parquet", FEAT_DIR, CHUNK)
print(f"lighting/jpegrand: {(time.time() - t0) / 60:.1f} min")
del est; gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
phys_aug = pd.concat([geo_aug[G.FEATURE_NAMES].reset_index(drop=True),
                      light_aug[Lg.FEATURE_NAMES].reset_index(drop=True)], axis=1)

## 4 · Train
* **JPEG-aug** models: trained on clean + compressed copies of every training image.
* **Clean-trained** models: notebook 02's detector, for reference.

In [ ]:
y_tr = manifest.label.values[tr_idx]
clip_tr = np.r_[clip_clean[tr_idx], clip_aug]
phys_tr = pd.concat([phys_clean.iloc[tr_idx].reset_index(drop=True), phys_aug], ignore_index=True)
y2 = np.r_[y_tr, y_tr]
q2 = np.r_[np.full(len(tr_idx), 96.0), q_aug]          # 96 = no processing applied by us
groups = np.r_[np.arange(len(tr_idx)), np.arange(len(tr_idx))]
t0 = time.time()
robust = M.RobustDetector(PHYS_COLS, seed=SEED).fit(clip_tr, phys_tr, y2, q2, groups)
print(f"trained in {(time.time() - t0) / 60:.1f} min")
clean_det = joblib.load(nb02 / "models" / "detector.joblib")["detector"]

## 5 · Evaluate on unseen generators (clean and processed test images)

In [ ]:
val_manifest = manifest.iloc[val_idx].reset_index(drop=True)
Q_TEST = {"clean": 96.0, "jpeg75": 75.0, "jpeg50": 50.0, "down0.5": 96.0}
N = M.RobustDetector.NAMES
KEY_MODELS = ["CLIP (clean-trained)", "Fused (clean-trained)", N["clip"], N["phys"], N["fused"], N["fused_q"]]
PAIRS = [(N["fused_q"], N["clip"]), (N["fused"], N["clip"]),
         (N["clip"], "CLIP (clean-trained)"), (N["fused_q"], "CLIP (clean-trained)")]

all_scores, tables, boots = {}, [], {}
for cond, (c, ph) in conds.items():
    q = np.full(len(c), Q_TEST[cond])
    s = robust.predict(c, ph, q)
    s0 = clean_det.predict(c, ph)
    s["CLIP (clean-trained)"] = s0[M.BASELINE]
    s["Fused (clean-trained)"] = s0["Fused: CLIP+geometry+lighting"]
    s = {k: s[k] for k in KEY_MODELS}
    all_scores[cond] = s
    r = P.evaluate(s, val_manifest, TRAIN_GEN)
    tables.append(r.assign(condition=cond))
    t0 = time.time()
    boots[cond] = P.bootstrap_mean_auc(s, val_manifest, TRAIN_GEN, PAIRS, n_boot=N_BOOT, seed=SEED)
    print(f"{cond}: bootstrap {time.time() - t0:.0f}s")
res = pd.concat(tables)
unseen = res[~res.seen]
auc_tab = unseen.groupby(["model", "condition"]).auc.mean().unstack()[["clean"] + PERTURBATIONS].loc[KEY_MODELS]
acc_tab = unseen.groupby(["model", "condition"]).bal_acc.mean().unstack()[["clean"] + PERTURBATIONS].loc[KEY_MODELS]
print("Mean AUC over unseen generators")
display(auc_tab.round(4))
print("Mean balanced accuracy over unseen generators")
display(acc_tab.round(4))

In [ ]:
rows = []
for cond, (_, diffs) in boots.items():
    for name, (d, lo, hi, p) in diffs.items():
        rows.append({"condition": cond, "comparison": name, "diff": d, "ci_lo": lo, "ci_hi": hi,
                     "P(diff<=0)": p, "significant": lo > 0 or hi < 0})
diff_tab = pd.DataFrame(rows)
print("Paired bootstrap, mean AUC over unseen generators (95% CI)")
display(diff_tab.round(4))
print("Per-generator AUC at JPEG q50")
display(P.summary_table(res[res.condition == "jpeg50"])[KEY_MODELS].round(4))

In [ ]:
import matplotlib.pyplot as plt
show = ["CLIP (clean-trained)", N["clip"], N["phys"], N["fused_q"]]
fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [2.2, 1]})
ax = axes[0]
x = np.arange(len(conds)); w = 0.2
for i, mname in enumerate(show):
    pts = np.array([boots[c][0][mname] for c in conds])           # [mean, lo, hi]
    bars = ax.bar(x + (i - 1.5) * w, pts[:, 0], w, yerr=[pts[:, 0] - pts[:, 1], pts[:, 2] - pts[:, 0]],
                  capsize=3, label=mname)
    ax.bar_label(bars, fmt="%.3f", fontsize=7, padding=2)
ax.set_xticks(x, [c if c != "down0.5" else "down0.5\n(never seen in training)" for c in conds])
ax.set_ylim(0.5, 1.02); ax.grid(axis="y", alpha=.3)
ax.set_title("Mean AUC on unseen generators (95% bootstrap CI)"); ax.legend(fontsize=8, loc="lower left")

ax = axes[1]
qq = np.linspace(50, 100, 51)
wc, wp = robust.branch_weights(qq)
ax.plot(qq, wc, label="weight on CLIP logit"); ax.plot(qq, wp, label="weight on physics logit")
ax.invert_xaxis(); ax.set_xlabel("JPEG quality (← more compression)"); ax.grid(alpha=.3)
ax.set_title("Quality-aware fusion: how much it trusts each branch"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT_DIR / "robustness_aug.png", dpi=130); plt.show()

## 6 · Save

In [ ]:
joblib.dump({"detector": robust, "train_generator": TRAIN_GEN, "clip_model": CLIP_MODEL,
             "normal_backend": NORMAL_BACKEND, "normal_res": NORMAL_RES, "normal_steps": NORMAL_STEPS,
             "normalise_format": NORMALISE_FORMAT, "phys_cols": PHYS_COLS, "jpegrand_range": I.JPEGRAND_RANGE},
            OUT_DIR / "models" / "detector_robust.joblib")
res.to_csv(OUT_DIR / "results_aug.csv", index=False)
diff_tab.to_csv(OUT_DIR / "bootstrap_diffs.csv", index=False)
wc, wp = robust.branch_weights([96, 75, 50])
out = {
    "train_generator": TRAIN_GEN, "n_boot": N_BOOT, "clip_C": robust.C,
    "stack_quality_aware_coef[clip,phys,t,t*clip,t*phys]": robust.stack_q.coef_[0].round(4).tolist(),
    "branch_weights_at_q96_75_50": {"clip": np.round(wc, 3).tolist(), "physics": np.round(wp, 3).tolist()},
    "mean_auc_unseen": auc_tab.round(4).to_dict(),
    "mean_bal_acc_unseen": acc_tab.round(4).to_dict(),
    "bootstrap_mean_auc": {c: {k: np.round(v, 4).tolist() for k, v in b[0].items()} for c, b in boots.items()},
    "bootstrap_diffs": {c: {k: np.round(v, 4).tolist() for k, v in b[1].items()} for c, b in boots.items()},
    "jpeg50_auc": P.summary_table(res[res.condition == "jpeg50"])[KEY_MODELS].round(4).to_dict(),
}
(OUT_DIR / "results_aug.json").write_text(json.dumps(out, indent=2, default=str))
print(json.dumps({"mean_auc_unseen": out["mean_auc_unseen"],
                  "branch_weights_at_q96_75_50": out["branch_weights_at_q96_75_50"]}, indent=1))